# Session 4: SELECT Statement Basics

This interactive notebook covers data query and retrieval fundamentals in SQL: column projection, row limitation with `LIMIT`, deduplication with `DISTINCT`, readability improvements with `AS` column aliases, and syntactic clause order debugging.

---

## Overview of Tasks
- **Task 1:** Create `MusicPlaylist` table (`id`, `song_name`, `artist`, `genre`, `duration`), insert 5+ tracks, and execute `SELECT *`.
- **Task 2:** Project only `song_name` and `artist`, fetching the first 3 rows using `LIMIT 3`.
- **Task 3:** Create `FoodOrders` table (`id`, `restaurant`, `food_item`, `order_date`) and extract unique restaurants using `DISTINCT`.
- **Task 4:** Rename projected columns using `AS` aliases (`Dish` and `Date Ordered`).
- **Task 5:** Diagnostic debugging of queries combining `DISTINCT` and `LIMIT`, fixing clause ordering and semantic issues.

### Task 1: Create 'MusicPlaylist' Table & Retrieve All Columns

In [ ]:
import sqlite3
import os
import pandas as pd

os.makedirs("database", exist_ok=True)
db_path = os.path.join("database", "music_and_orders.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS MusicPlaylist;")
create_music_sql = """
CREATE TABLE MusicPlaylist (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    song_name VARCHAR(100) NOT NULL,
    artist VARCHAR(100) NOT NULL,
    genre VARCHAR(50) NOT NULL,
    duration INT NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""
cursor.execute(create_music_sql)

sample_songs = [
    ("Starboy", "The Weeknd", "Synth-pop / R&B", 230),
    ("Blinding Lights", "The Weeknd", "Synth-wave", 200),
    ("Tum Hi Ho", "Arijit Singh", "Bollywood Romantic", 262),
    ("Brown Munde", "AP Dhillon", "Punjabi Hip-Hop", 265),
    ("Shape of You", "Ed Sheeran", "Pop", 233),
    ("Kesariya", "Arijit Singh", "Bollywood", 268)
]
cursor.executemany(
    "INSERT INTO MusicPlaylist (song_name, artist, genre, duration) VALUES (?, ?, ?, ?);",
    sample_songs
)
conn.commit()

df_all_songs = pd.read_sql_query("SELECT * FROM MusicPlaylist;", conn)
display(df_all_songs)

### Task 2: Project 'song_name' & 'artist' with LIMIT 3
Selecting only desired columns reduces network bandwidth and query execution time.

In [ ]:
query_limit = "SELECT song_name, artist FROM MusicPlaylist LIMIT 3;"
df_top3 = pd.read_sql_query(query_limit, conn)
display(df_top3)

### Task 3: Create 'FoodOrders' Table & Query Unique Restaurants with DISTINCT
`DISTINCT` eliminates duplicate rows in the query result set.

In [ ]:
cursor.execute("DROP TABLE IF EXISTS FoodOrders;")
create_orders_sql = """
CREATE TABLE FoodOrders (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    restaurant VARCHAR(100) NOT NULL,
    food_item VARCHAR(100) NOT NULL,
    order_date DATE NOT NULL
);
"""
cursor.execute(create_orders_sql)

sample_orders = [
    ("Haldiram's", "Chole Bhature", "2026-10-20"),
    ("Paradise Biryani", "Chicken Biryani", "2026-10-22"),
    ("Haldiram's", "Raj Kachori", "2026-10-25"),
    ("Glen's Bakehouse", "Red Velvet Cupcake", "2026-10-28"),
    ("Paradise Biryani", "Mutton Biryani", "2026-10-30"),
    ("Bademiya", "Chicken Baida Roti", "2026-11-01"),
    ("Haldiram's", "Gulab Jamun", "2026-11-02")
]
cursor.executemany(
    "INSERT INTO FoodOrders (restaurant, food_item, order_date) VALUES (?, ?, ?);",
    sample_orders
)
conn.commit()

df_unique_restaurants = pd.read_sql_query("SELECT DISTINCT restaurant FROM FoodOrders;", conn)
display(df_unique_restaurants)

### Task 4: Column Aliases ('Dish' and 'Date Ordered')
Using `AS` gives readable business names to table columns in analytical outputs.

In [ ]:
query_alias = 'SELECT food_item AS "Dish", order_date AS "Date Ordered" FROM FoodOrders;'
df_aliased = pd.read_sql_query(query_alias, conn)
display(df_aliased)

### Task 5: Debugging Queries Combining DISTINCT and LIMIT

#### The Problem:
Consider: `SELECT DISTINCT food_item, restaurant FROM FoodOrders LIMIT 2;`

1. **Clause Order / Misplacement:** A common student mistake is placing `LIMIT` before `FROM`:  
   `SELECT DISTINCT food_item, restaurant LIMIT 2 FROM FoodOrders;` $\rightarrow$ **Syntax Error!**
2. **Multi-Column Scope:** `DISTINCT` applies to the combination `(food_item, restaurant)`, not just one column.
3. **Determinism:** `LIMIT` without `ORDER BY` returns non-deterministic rows depending on physical storage layout.

In [ ]:
# Demonstration of misplaced LIMIT causing syntax error:
try:
    cursor.execute("SELECT DISTINCT food_item, restaurant LIMIT 2 FROM FoodOrders;")
except sqlite3.OperationalError as e:
    print(f"Caught Syntax Error: {e}")

# Corrected Deterministic Query:
corrected_query = """
SELECT DISTINCT food_item, restaurant 
FROM FoodOrders 
ORDER BY restaurant, food_item 
LIMIT 2;
"""
df_fixed = pd.read_sql_query(corrected_query, conn)
display(df_fixed)

conn.close()